In [15]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [16]:
class BatsmanState(TypedDict):              #inherits from TypedDict 
    runs: int
    balls: int
    fours: int
    sixes: int
    strike_rate: float
    balls_per_boundary: float 
    boundary_percent: float
    summary: str

In [17]:
def calculate_strike_rate(state: BatsmanState) -> BatsmanState:
    strike_rate = (state['runs'] / state['balls']) * 100
    return {'strike_rate': strike_rate}                                                             #partial update
    
def calculate_balls_per_boundary(state: BatsmanState) -> BatsmanState:
    balls_per_boundary = state['balls'] / (state['fours'] + state['sixes'])
    return {'balls_per_boundary': balls_per_boundary}

def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
    #calculate percentage of runs made from hitting boundaries
    boundary_percent = ((state['fours'] * 4 + state['sixes'] * 6) / state['runs']) / 100
    return {'boundary_percent': boundary_percent}                                                   #partial update

def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""
Strike Rate = {state['strike_rate']} \n
Balls per Boundary = {state['balls_per_boundary']} \n
Boundary Percent = {state['boundary_percent']} \n
"""
    return {'summary': summary}                                                                     #partial update

In [18]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_strike_rate', calculate_strike_rate)
graph.add_node('calculate_balls_per_boundary', calculate_balls_per_boundary)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_strike_rate')
graph.add_edge(START, 'calculate_balls_per_boundary')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_strike_rate', 'summary')
graph.add_edge('calculate_balls_per_boundary', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()

In [19]:
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

final_state = workflow.invoke(initial_state)

In [20]:
final_state

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'strike_rate': 200.0,
 'balls_per_boundary': 5.0,
 'boundary_percent': 0.0048,
 'summary': '\nStrike Rate = 200.0 \n\nBalls per Boundary = 5.0 \n\nBoundary Percent = 0.0048 \n\n'}